# Telescope Runs — Stage 1 Demo (site / sun-event helper)

This notebook demonstrates `solsys_code/telescope_runs.py` (issue #37 Stage 1), the
site/ephemeris helper that resolves a telescope name to its `Observatory` record
and computes dip-corrected UTC sun-event times for a given date.

It exercises three helpers:

- `get_site(name)` — resolves a telescope name (a key of `SITES`) to an
  `Observatory` model record.
- `horizon_dip(altitude_m)` — computes the horizon dip correction (in degrees)
  for an observer at the given altitude.
- `sun_event(site, date, kind)` — returns (setting, rising) UTC `Time` objects
  for either the dip-corrected sun event (`kind='sun'`) or the -15°
  astronomical-dark window (`kind='dark'`).

This notebook lives in `pre_executed/` because it is **DB-dependent** (it needs
`Observatory` records to exist) and is therefore **NOT run during
Sphinx/CI/ReadTheDocs builds**, per `docs/notebooks/README.md`'s guidance to put
notebooks that need resources not guaranteed in every environment under
`pre_executed/`.

## Prerequisites

This notebook requires `Observatory` records for MPC obscodes `'268'`
(Magellan-Clay), `'269'` (Magellan-Baade), `'809'` (NTT / La Silla), and
`'E10'` (FTS / Siding Spring) to exist in the database. These can be created
via the `CreateObservatory` form (or an equivalent fixture/migration).

The site coordinates and timezones used below are documented in
`solsys_code/tests/test_telescope_runs.py`'s `setUpTestData`:

| obscode | short_name | lat | lon | altitude (m) | timezone |
|---|---|---|---|---|---|
| 268 | Magellan-Clay | -29.0146 | -70.6926 | 2402 | America/Santiago |
| 269 | Magellan-Baade | -29.0146 | -70.6926 | 2402 | America/Santiago |
| 809 | NTT | -29.2567 | -70.7300 | 2347 | America/Santiago |
| E10 | FTS | -31.2734 | 149.0612 | 1149 | Australia/Sydney |

In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
repo_root = str(Path.cwd().resolve().parents[2])
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

In [2]:
from datetime import date

from solsys_code.telescope_runs import SITES, get_site, horizon_dip, sun_event

print(SITES)

{'Magellan-Clay': '268', 'Magellan-Baade': '269', 'NTT': '809', 'FTS': 'E10'}


## Resolving a telescope to its site

`get_site(name)` looks up `name` in `SITES` to get an MPC obscode, then returns
the corresponding `Observatory` record. This is the single source of truth for
the site's geodetic position (lat/lon/altitude) and IANA timezone.

In [3]:
site = get_site('NTT')

print('short_name:', site.short_name)
print('obscode:', site.obscode)
print('lat:', site.lat)
print('lon:', site.lon)
print('altitude:', site.altitude)
print('timezone:', site.timezone)

short_name: European Southern Observatory, La Silla
obscode: 809
lat: -29.25881957385624
lon: -70.73374000000003
altitude: 2345.365187853709
timezone: America/Santiago


## Horizon dip

An observer above sea level sees the true horizon dip below the astronomical
horizon. The correction is:

```
dip = 1.76' * sqrt(altitude_m)   (in arcminutes, converted to degrees)
```

This is added to the standard refraction + solar semi-diameter offset
(0.833°) when computing sunset/sunrise, so higher sites see the Sun set
slightly later (and rise slightly earlier) than sea-level observers.

In [4]:
dip = horizon_dip(site.altitude)
print(f'horizon dip at {site.altitude:.0f} m: {dip:.4f} deg')

# Reference (EPHEM-03): at Las Campanas (2402 m), dip should be 1.44 deg +/- 0.02.

horizon dip at 2345 m: 85.2350 arcmin deg


## Sun events

`sun_event(site, date, kind)` returns `(setting, rising)` as UTC `astropy.time.Time`
objects for the observing night that starts on the evening of `date`:

- `kind='sun'` — dip-corrected sunset/sunrise (solar altitude crosses
  `-(0.833 + dip)` degrees).
- `kind='dark'` — astronomical-dark window (solar altitude crosses -15
  degrees, no dip correction).

In [5]:
sample_date = date(2026, 6, 10)

sunset, sunrise = sun_event(site, sample_date, 'sun')
dark_start, dark_end = sun_event(site, sample_date, 'dark')

print('sunset    :', sunset.iso)
print('sunrise   :', sunrise.iso)
print('dark start:', dark_start.iso)
print('dark end  :', dark_end.iso)

sunset    : 2026-06-10 21:58:45.352
sunrise   : 2026-06-11 11:26:24.434
dark start: 2026-06-10 23:01:43.008
dark end  : 2026-06-11 10:23:24.844


## Parsing run lines with optional partial-night window tokens

`parse_run_line` accepts an optional trailing `(BoN|HHMM)-(EoN|HHMM)` token that
restricts the observing window to a portion of the night.  The token is stored in
the `start_window` and `end_window` fields of the returned `ParsedRun`.

- `BoN` — beginning of night (computed sunset)
- `EoN` — end of night (computed sunrise)
- `HHMM` — a fixed UTC time: if HHMM < 1200 it is treated as next-morning UTC
  (evening_date + 1 day); if HHMM ≥ 1200 it is evening-date UTC.

When `start_window` / `end_window` are `None` the full night (sunset → sunrise) is used.

In [6]:
from solsys_code.telescope_runs import parse_run_line

# Full night — start_window and end_window default to None
full = parse_run_line('NTT EFOSC2 allocation 9-13 July')
print('Full night:')
print(f'  start_window={full.start_window!r}  end_window={full.end_window!r}')

# First half of night — BoN (sunset) to 06:26 UTC next morning
first_half = parse_run_line('Magellan-Clay Lightspeed 18-20 July BoN-0626')
print('\nFirst-half night (BoN-0626):')
print(f'  start_window={first_half.start_window!r}  end_window={first_half.end_window!r}')

# Second half of night — 06:46 UTC next morning to EoN (sunrise)
second_half = parse_run_line('Magellan-Clay LDSS3 18-20 July 0646-EoN')
print('\nSecond-half night (0646-EoN):')
print(f'  start_window={second_half.start_window!r}  end_window={second_half.end_window!r}')

Full night:
  start_window=None  end_window=None

First-half night (BoN-0626):
  start_window='BoN'  end_window='0626'

Second-half night (0646-EoN):
  start_window='0646'  end_window='EoN'


## Closing notes

For Las Campanas (Magellan-Clay / Magellan-Baade, obscodes 268/269), the computed
sun-event times for June 2026 match the LCO skycalc reference tool to within 2
minutes (EPHEM-04). NTT (La Silla, obscode 809) is part of the same site cluster
and uses the same `America/Santiago` timezone.

This notebook is **pre-executed** and intentionally excluded from automated doc
builds (it is not referenced in `docs/notebooks.rst`) because it depends on
`Observatory` records existing in the database.